# Predict Podcast Listening Time
Kaggle Competition Link: https://www.kaggle.com/competitions/playground-series-s5e4/overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

## Data Loading

In [ ]:
# Loading Training and Testing Datasets and Sample Submission File from links
df =  pd.read_csv("https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-podcast-listening-time/train.csv")
df_aug  = pd.read_csv("https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-podcast-listening-time/podcast_dataset.csv")
df_test = pd.read_csv("https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-podcast-listening-time/test.csv")
df_sample = pd.read_csv("https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-podcast-listening-time/sample_submission.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Check unique element for each column
for col in df.columns:
    print(f"{col}: {df[col].unique()}")

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

## Data Cleaning

In [ ]:
# Check rows where Episode_Length_minutes > 300
df_test[df_test["Episode_Length_minutes"] > 300]

In [ ]:
# Change the value for rows with Episode_Length_minutes with the mean of Episode_Length_minutes based on the Podcast Name excluding such rows
df_test.loc[df_test["Episode_Length_minutes"] > 300, "Episode_Length_minutes"] = df_test.loc[df_test["Episode_Length_minutes"] > 300, "Podcast_Name"].map(df.groupby("Podcast_Name")["Episode_Length_minutes"].mean())

In [ ]:
df_test.iloc[[54434, 56597]]

In [ ]:
# Create column that distinguishes train data with '1' or '0' for test. Combine datasets after
df["is_train"] = 1
df_aug["is_train"] = 1
df_test["is_train"] = 0
df_combined = pd.concat([df, df_test, df_aug])

In [ ]:
df_combined.head()

In [ ]:
df_combined.describe()

In [ ]:
# Fill null values for Episode_Length_minutes, Guest_Popularity_percentage, and Number_of_Ads based on Podcast Name
df_combined["Episode_Length_minutes"] = df_combined.groupby("Podcast_Name")["Episode_Length_minutes"].transform(lambda x: x.fillna(x.mean()))
df_combined["Guest_Popularity_percentage"] = df_combined.groupby("Podcast_Name")["Guest_Popularity_percentage"].transform(lambda x: x.fillna(x.mean()))
df_combined["Number_of_Ads"] = df_combined.groupby("Podcast_Name")["Number_of_Ads"].transform(lambda x: x.fillna(x.mean()))

In [ ]:
df_combined.describe()

In [ ]:
df_combined.duplicated().sum()

In [ ]:
df_combined.drop_duplicates(inplace=True)

In [ ]:
df_combined.info()

In [ ]:
numerical_cols = df_combined.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_combined.select_dtypes(include=[object]).columns.tolist()
# Remove is_train, id
numerical_cols.remove("is_train")
numerical_cols.remove("id")
numerical_cols.remove("Listening_Time_minutes")

In [ ]:
numerical_cols

## Exploratory Data Analysis

In [ ]:
# Box Plot of listening time based on Genre
fig = px.box(df_combined, x="Genre", y="Listening_Time_minutes", title="Listening Time by Genre")
fig.show()

In [ ]:
# Box Plot of listening time based on Episode_sentiment
fig = px.box(df_combined, x="Episode_Sentiment", y="Listening_Time_minutes", title="Listening Time by Episode_Sentiment")
fig.show()

In [ ]:
# Box Plot of Listening Time based on Publication_Day
fig = px.box(df_combined, x="Publication_Day", y="Listening_Time_minutes", title="Listening Time by Publication_Day")
fig.show()

In [ ]:
# Box Plot Listening Time Based on Publication_Time
fig = px.box(df_combined, x="Publication_Time", y="Listening_Time_minutes", title="Listening Time by Publication_Time")
fig.show()

In [ ]:
# Box Plot Listening Time based on Episode_Title
fig = px.box(df_combined, x="Episode_Title", y="Listening_Time_minutes", title="Listening Time by Episode_Title")
fig.show()

In [ ]:
# Box Plot Listening Time based on Podcast_Name
fig = px.box(df_combined, x="Podcast_Name", y="Listening_Time_minutes", title="Listening Time by Podcast_Name")
fig.show()

In [ ]:
# Scatter plot of listening time against Host_popularity_percentage
fig = px.scatter(df_combined, x="Host_Popularity_percentage", y="Listening_Time_minutes", title="Listening Time vs Host Popularity")
fig.show()

In [ ]:
# Scatter plot of listening time against Guest_popularity_percentage
fig = px.scatter(df_combined, x="Guest_Popularity_percentage", y="Listening_Time_minutes", title="Listening Time vs Guest Popularity")
fig.show()

## Feature Engineering

In [ ]:
# Scale Numerical Columns
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_combined[numerical_cols] = scaler.fit_transform(df_combined[numerical_cols])

**Label Encoding Treatments**
- Nominal Encoding: Podcast_Name, Genre
- Ordinal Encoding: Episode_Title, Publication_Day, Publication _Time, Episode_Sentiment

In [ ]:
# Label Encoder for Nominal Encoding
le = LabelEncoder()
df_combined["Genre"] = le.fit_transform(df_combined["Genre"])

In [ ]:
# Remove Episode string and convert string to int
df_combined["Episode_Title"] = df_combined["Episode_Title"].str.replace("Episode", "").astype(int)

In [ ]:
# Check unique values for Publication_Day, Publication_time, and Episode_Sentiment
print(df_combined["Publication_Day"].unique())
print(df_combined["Publication_Time"].unique())
print(df_combined["Episode_Sentiment"].unique())

In [ ]:
# Set Monday as 1, Tuesday as 2, and so on
df_combined["Publication_Day"] = df_combined["Publication_Day"].map({"Monday": 1, "Tuesday": 2, "Wednesday": 3, "Thursday": 4, "Friday": 5, "Saturday": 6, "Sunday": 7})

In [ ]:
# Set Morning as 1, Afternoon as 2, Evening as 3, and Night as 4
df_combined["Publication_Time"] = df_combined["Publication_Time"].map({"Morning": 1, "Afternoon": 2, "Evening": 3, "Night": 4})

In [ ]:
# Set Negative as 0, Neutral as 1, and Positive as 2
df_combined["Episode_Sentiment"] = df_combined["Episode_Sentiment"].map({"Negative": 0, "Neutral": 1, "Positive": 2})

In [ ]:
from sklearn.model_selection import KFold
def target_encode(df, col, target, n_splits=5):
    df = df.copy()
    global_mean = df[target].mean()
    df[f'{col}_target_enc'] = np.nan

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    for train_idx, val_idx in kf.split(df):
        train_df, val_df = df.iloc[train_idx], df.iloc[val_idx]
        means = train_df.groupby(col)[target].mean()
        df.loc[df.index[val_idx], f'{col}_target_enc'] = df.loc[df.index[val_idx], col].map(means)

    # Fill unknowns (unseen in training folds) with global mean
    df[f'{col}_target_enc'] = df[f'{col}_target_enc'].fillna(global_mean)
    return df

In [ ]:
df_combined = target_encode(df_combined, col='Podcast_Name', target='Listening_Time_minutes')

In [ ]:
# Drop Podcast_Name
df_combined = df_combined.drop(["Podcast_Name"], axis=1)

In [ ]:
df_combined.head()

In [ ]:
'''target_means = df_combined.groupby('Podcast_Name')['Listening_Time_minutes'].mean()
df_combined['podcast_target_enc'] = df_combined['Podcast_Name'].map(target_means)'''

In [ ]:
# Conduct PCA
from sklearn.decomposition import PCA

In [ ]:
# Show Scree Plot
def conduct_PCA(df):
  df_pca = df.copy()
  df_pca.dropna(inplace=True)
  print(f"PCA Conduct")
  pca = PCA(n_components=df_pca.shape[1])
  df_pca = pca.fit_transform(df_pca)
  cumsum = np.cumsum(pca.explained_variance_ratio_)
  print(cumsum)
  d = np.argmax(cumsum >= 0.90) + 1
  print("dimension : ", d)

  explained_variance = pca.explained_variance_ratio_
  cumulative_variance = np.cumsum(explained_variance)
  plt.plot(range(1, len(explained_variance) + 1), cumulative_variance)
  plt.xlabel('Number of Components')
  plt.ylabel('Cumulative Explained Variance')
  plt.title(f'Scree Plot for DataFrame')
  plt.show()

In [ ]:
conduct_PCA(df_combined)

In [ ]:
# Separate dataset into df and df_test based on df_train
df = df_combined[df_combined["is_train"] == 1]
df_test = df_combined[df_combined["is_train"] == 0]

In [ ]:
df_test.info()

In [ ]:
df.isna().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isna().sum()

## Training and Test Splits

In [ ]:
# Keep the following columns for X ['Podcast_Name', 'Episode_Length_minutes', 'Genre', 'Host_Popularity_percentage', 'Publication_Day', 'Publication_Time', 'Guest_Popularity_percentage', 'Number_of_Ads', 'Episode_Sentiment']
X = df.drop(["Listening_Time_minutes", "is_train", "id"], axis=1)
y = df["Listening_Time_minutes"]

## Feature Selection

In [ ]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
'''
model_rf = RandomForestRegressor(random_state=42)
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

grid_search = GridSearchCV(
    estimator=model_rf,
    param_grid=param_grid,
    cv=5,  # 5-fold cross-validation
    n_jobs=-1,  # Use all available CPU cores
    verbose=2
)

grid_search.fit(X, y)
print(grid_search.best_params_)'''

In [ ]:
model_rf = RandomForestRegressor(max_depth= None, min_samples_leaf = 1, min_samples_split = 2, n_estimators= 100)
model_rf.fit(X, y)

In [ ]:
sel_sfm = SelectFromModel(model_rf, prefit=True)
sel_sfm_index = sel_sfm.get_support()
selected_rfe_features = X.iloc[:, sel_sfm_index]
print(selected_rfe_features.columns)

In [ ]:
# Check accuracy of model
from sklearn.metrics import mean_squared_error, r2_score

y_pred = model_rf.predict(X)
mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

print("Mean Squared Error:", mse)
rmse = mse ** 0.5
print("Root Mean Squared Error:", rmse)
print("R-squared:", r2)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.feature_selection import RFECV, mutual_info_regression, SelectKBest, f_regression
from sklearn.model_selection import KFold
# Use Regression Model for Feature Selection
def rfevc_features_selected(X, y, features_to_select):
  print(f'Test DF')
  min_features_to_select = features_to_select  # Minimum number of features to consider
  clf = Ridge()
  cv = KFold(5) # Changed to KFold which is suitable for regression

  rfecv = RFECV(
      estimator=clf,
      step=1,
      cv=cv,
      scoring="neg_mean_squared_error", # Changed scoring metric to a suitable one for Regression
      min_features_to_select=min_features_to_select,
      n_jobs=2,
  )
  rfecv.fit(X, y)
  selected_inc_features = X.columns[rfecv.support_]
  print(f"Optimal number of features: {rfecv.n_features_}")
  return selected_inc_features

In [ ]:
rfevc_features_selected(X, y, 1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Model Training

## XGBoost Hyperparameter Tuning and Training

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor
# Hyperparameter space for XGBoostRegressor
param_dist = {
    "max_depth": range(3, 10),
    "min_child_weight": range(1, 8),
    "subsample": [0.6, 0.8, 1],
    "colsample_bytree": [0.6, 0.8, 1],
    "gamma": [0, 1, 5],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 200, 500],
}

search = RandomizedSearchCV(
    estimator=XGBRegressor(
    n_estimators = 1000,
    eval_metric = "rmse",
    objective = "reg:squarederror",
    early_stopping_rounds = 30,
    tree_method = "hist"
    ),
    param_distributions=param_dist,
    n_iter=50,
    cv=3,
    scoring="neg_mean_squared_error",
    verbose=1,
    n_jobs=-1
)

search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print(search.best_params_)

In [ ]:
# Implement LightGBM, XGBoost, and GBM as Model. Use Default Parameters to serve as baseline
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor
model = XGBRegressor(subsample = 0.8, n_estimators = 500, min_child_weight = 2, max_depth = 9, learning_rate = 0.05, gamma = 1, colsample_bytree = 0.8, eval_metric = "rmse", objective = "reg:squarederror", tree_method = "hist")
model.fit(X_train, y_train)

In [ ]:
model_rf = RandomForestRegressor(max_depth= None, min_samples_leaf = 1, min_samples_split = 2, n_estimators= 100)
model_rf.fit(X_train, y_train)

## Neural Network - Tensorflow

In [ ]:
import tensorflow as tf

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1)
])

In [ ]:
import tensorflow.keras.backend as K
def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

In [ ]:
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae', rmse])

In [ ]:
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))

In [ ]:
# Plot History of Neural Network - Regression Steps
def plot_history(history):
  fig, (ax1, ax2) = plt.subplots(2, figsize=(12, 12)) # Unpack into ax1 and ax2
  ax1.legend(['train', 'validation'], loc='upper left') # Call legend on ax1
  ax1.plot(history.history['loss']) # Plot on ax1
  ax1.plot(history.history['val_loss']) # Plot on ax1
  ax1.set_title('model loss') # Set title on ax1
  ax1.set_ylabel('loss') # Set label on ax1
  ax1.set_xlabel('epoch') # Set label on ax1

In [ ]:
plot_history(history)

## Model Evaluation

In [ ]:
# Check for Root Mean Square error and R2 Score
y_pred = model.predict(X_test)
print(f"Root Mean Square Error: {np.sqrt(mean_squared_error(y_test, y_pred))}")
print(f"R2 Score: {r2_score(y_test, y_pred)}")

## Preparing Submission File

In [ ]:
df_sample = pd.read_csv("https://raw.githubusercontent.com/machiwao/ml-development/refs/heads/main/kaggle/predict-podcast-listening-time/sample_submission.csv")

In [ ]:
id = df_sample.pop('id')
# df_test = df_test.drop(["Listening_Time_minutes", "is_train", "id"], axis=1)
y_pred = model_rf.predict(df_test)

# Reshape y_pred to be 1-dimensional
y_pred = y_pred.reshape(-1)  # or y_pred = y_pred.flatten()


# Create a submission DataFrame
submission_df = pd.DataFrame({
    'id': id,
    'Listening_Time_minutes': y_pred
})

# Save the submission DataFrame to a CSV file
submission_df.to_csv('submission_file.csv', index=False)
print("Submission file created: submission_file.csv")